# How to connect a chat bot to your memory service

In [16]:
import dotenv

dotenv.load_dotenv("../../.env", override=True)


True

In [17]:
from langgraph_sdk import get_client

# Update to your URL. Copy this from page of your LangGraph Deployment
deployment_url = "http://127.0.0.1:2024"

client = get_client(url=deployment_url)

## Example Chat Bot

The bot fetches user memories my semantic similarity, templates them, then responds!

In [93]:
import os
import uuid
from datetime import datetime, timezone
from typing import List, Optional

import langsmith
from langchain.chat_models import init_chat_model
from langchain_core.messages import AnyMessage
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableConfig
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import START, StateGraph, add_messages
from langgraph_sdk import get_client
from pydantic import BaseModel, Field
from typing_extensions import Annotated, TypedDict

from memory_agent import (
    constants,
    settings,
    utils,
)


class ChatState(TypedDict):
    """The state of the chatbot."""

    messages: Annotated[List[AnyMessage], add_messages]
    user_memories: List[dict]


class ChatConfigurable(TypedDict):
    """The configurable fields for the chatbot."""

    user_id: str
    thread_id: str
    memory_service_url: str = ""
    model: str
    delay: Optional[float]


def _ensure_configurable(config: RunnableConfig) -> ChatConfigurable:
    """Ensure the configuration is valid."""
    return ChatConfigurable(
        user_id=config["configurable"]["user_id"],
        thread_id=config["configurable"]["thread_id"],
        mem_assistant_id=config["configurable"]["mem_assistant_id"],
        memory_service_url=config["configurable"].get(
            "memory_service_url", os.environ.get("MEMORY_SERVICE_URL", "")
        ),
        model=config["configurable"].get(
            # "model", "accounts/fireworks/models/firefunction-v2"
            "model", "gpt-4o-mini"
        ),
        # delay=config["configurable"].get("delay", 60),
        delay=config["configurable"].get("delay", 1),
    )


PROMPT = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are a helpful and friendly chatbot. Get to know the user!"
            " Ask questions! Be spontaneous!"
            "{user_info}\n\nSystem Time: {time}",
        ),
        ("placeholder", "{messages}"),
    ]
).partial(
    time=lambda: datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S"),
)


@langsmith.traceable
def format_query(messages: List[AnyMessage]) -> str:
    """Format the query for the user's memories."""
    # This is quite naive :)
    return " ".join([str(m.content) for m in messages if m.type == "human"][-5:])


async def query_memories(state: ChatState, config: RunnableConfig) -> ChatState:
    """Query the user's memories."""
    print("query_memories")
    configurable: ChatConfigurable = config["configurable"]
    user_id = configurable["user_id"]
    index = utils.get_index()
    embeddings = utils.get_embeddings()

    query = format_query(state["messages"])
    vec = await embeddings.aembed_query(query)
    # You can also filter by memory type, etc. here.
    with langsmith.trace(
        "pinecone_query", inputs={"query": query, "user_id": user_id}
    ) as rt:
        response = index.query(
            vector=vec,
            filter={"user_id": {"$eq": str(user_id)}},
            include_metadata=True,
            top_k=10,
            namespace=settings.SETTINGS.pinecone_namespace,
        )
        rt.outputs["response"] = response
    memories = []
    if matches := response.get("matches"):
        memories = [m["metadata"][constants.PAYLOAD_KEY] for m in matches]
    return {
        "user_memories": memories,
    }


@langsmith.traceable
def format_memories(memories: List[dict]) -> str:
    """Format the user's memories."""
    if not memories:
        return ""
    # Note Bene: You can format better than this....
    memories = "\n".join(str(m) for m in memories)
    return f"""

## Memories

You have noted the following memorable events from previous interactions with the user.
<memories>
{memories}
</memories>
"""


async def bot(state: ChatState, config: RunnableConfig) -> ChatState:
    """Prompt the bot to resopnd to the user, incorporating memories (if provided)."""
    print("bot")
    configurable = _ensure_configurable(config)
    model = init_chat_model(configurable["model"])
    chain = PROMPT | model
    memories = format_memories(state["user_memories"])
    m = await chain.ainvoke(
        {
            "messages": state["messages"],
            "user_info": memories,
        },
        config,
    )

    return {
        "messages": [m],
    }


class MemorableEvent(BaseModel):
    """A memorable event."""

    description: str
    participants: List[str] = Field(
        description="Names of participants in the event and their relationship to the user."
    )


async def post_messages(state: ChatState, config: RunnableConfig) -> ChatState:
    print("post_messages")
    """Query the user's memories."""
    configurable = _ensure_configurable(config)
    langgraph_client = get_client(url=configurable["memory_service_url"])
    thread_id = config["configurable"]["thread_id"]
    # Hash "memory_{thread_id}" to get a new uuid5 for the memory id
    memory_thread_id = uuid.uuid5(uuid.NAMESPACE_URL, f"memory_{thread_id}")
    try:
        await langgraph_client.threads.get(thread_id=memory_thread_id)
    except Exception:
        await langgraph_client.threads.create(thread_id=memory_thread_id)

    await langgraph_client.runs.create(
        memory_thread_id,
        assistant_id=configurable["mem_assistant_id"],
        input={
            "messages": state["messages"],  # the service dedupes messages
        },
        config={
            "configurable": {
                "user_id": configurable["user_id"],
            },
        },
        # multitask_strategy="rollback",
        multitask_strategy="enqueue",
    )

    return {
        "messages": [],
    }


builder = StateGraph(ChatState, ChatConfigurable)
builder.add_node(query_memories)
builder.add_node(bot)
builder.add_node(post_messages)
builder.add_edge(START, "query_memories")
builder.add_edge("query_memories", "bot")
builder.add_edge("bot", "post_messages")

chat_graph = builder.compile(checkpointer=MemorySaver())

In [94]:
result = await client.assistants.search()
result

[{'assistant_id': 'eb11b265-df51-40a3-a345-5bfc6b61ab24',
  'graph_id': 'memory',
  'config': {'configurable': {'delay': 4,
    'schemas': {'MemorableEvent': {'system_prompt': "Extract any memorable events from the user's messages that you would like to remember.",
      'update_mode': 'insert',
      'function': {'description': 'A memorable event.',
       'properties': {'description': {'title': 'Description',
         'type': 'string'},
        'participants': {'description': 'Names of participants in the event and their relationship to the user.',
         'items': {'type': 'string'},
         'title': 'Participants',
         'type': 'array'}},
       'required': ['description', 'participants'],
       'title': 'MemorableEvent',
       'type': 'object'}}}}},
  'metadata': {},
  'name': 'Untitled',
  'created_at': '2025-02-21T13:37:36.248564+00:00',
  'updated_at': '2025-02-21T13:37:36.248564+00:00',
  'version': 1},
 {'assistant_id': 'a86c8600-ddcb-4f1c-ae39-b860c01c3dc0',
  'graph

In [95]:
mem_assistant = await client.assistants.create(
    graph_id="memory",
    config={
        "configurable": {
            "delay": 4,  # seconds wait before considering a thread as "completed"
            "schemas": {
                "MemorableEvent": {
                    "system_prompt": "Extract any memorable events from the user's"
                    " messages that you would like to remember.",
                    "update_mode": "insert",
                    "function": MemorableEvent.schema(),
                },
            },
        }
    },
)

/var/folders/q_/nkt84m3j2nb1ppf5cn3v6zq00000gn/T/ipykernel_92158/1171363277.py:11: PydanticDeprecatedSince20: The `schema` method is deprecated; use `model_json_schema` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.10/migration/
  "function": MemorableEvent.schema(),


In [96]:
mem_assistant = (await client.assistants.search(graph_id="memory"))[0]
mem_assistant

{'assistant_id': '9d5a371c-abd2-4dd4-a1dd-21fb98dad205',
 'graph_id': 'memory',
 'config': {'configurable': {'delay': 4,
   'schemas': {'MemorableEvent': {'system_prompt': "Extract any memorable events from the user's messages that you would like to remember.",
     'update_mode': 'insert',
     'function': {'description': 'A memorable event.',
      'properties': {'description': {'title': 'Description', 'type': 'string'},
       'participants': {'description': 'Names of participants in the event and their relationship to the user.',
        'items': {'type': 'string'},
        'title': 'Participants',
        'type': 'array'}},
      'required': ['description', 'participants'],
      'title': 'MemorableEvent',
      'type': 'object'}}}}},
 'metadata': {},
 'name': 'Untitled',
 'created_at': '2025-02-21T13:40:19.799576+00:00',
 'updated_at': '2025-02-21T13:40:19.799576+00:00',
 'version': 1}

In [97]:
import uuid

user_id = str(uuid.uuid4())  # more permanent

In [98]:
thread_id = str(uuid.uuid4())  # can adjust
result = await client.threads.create(thread_id=thread_id)
result

{'thread_id': 'f2020bda-37a6-4f8a-a9b4-510ed8a9b4f4',
 'created_at': '2025-02-21T13:40:25.224654+00:00',
 'updated_at': '2025-02-21T13:40:25.224658+00:00',
 'metadata': {},
 'status': 'idle',
 'config': {},
 'values': None}

In [100]:
class Chat:
    def __init__(self, user_id: str, thread_id: str):
        self.thread_id = thread_id
        self.user_id = user_id

    async def __call__(self, query: str) -> str:
        chunks = chat_graph.astream_events(
            input={
                "messages": [("user", query)],
            },
            config={
                "configurable": {
                    "user_id": self.user_id,
                    "thread_id": self.thread_id,
                    "memory_service_url": deployment_url,
                    "mem_assistant_id": mem_assistant["assistant_id"],
                    "delay": 4,
                }
            },
            version="v2",
        )
        res = ""
        async for event in chunks:
            if event.get("event") == "on_chat_model_stream":
                tok = event["data"]["chunk"].content
                print(tok, end="")
                res += tok
        return res

In [101]:
chat = Chat(user_id, thread_id)
chat

In [102]:
_ = await chat("Hi there")

query_memories
bot
Hi! How's your day going so far?post_messages


In [103]:
_ = await chat(
    "I've been planning a surprise party for my friend steve. "
    "He has been having a rough month and I want it to be special."
)

query_memories
bot
That sounds like a wonderful idea! It's great that you want to lift Steve's spirits. Do you have any specific themes or activities in mind for the party?post_messages


In [83]:
_ = await chat(
    "Steve really likes crocheting. Maybe I can do something with that? Or is that dumb... "
)

query_memories
bot
Not dumb at all! Incorporating something Steve loves, like crocheting, would make the surprise even more special and personal. You could have a crochet-themed party where everyone brings their own projects to work on, or even set up some fun crochet challenges! What do you think?post_messages


In [45]:
_ = await chat("He's also into capoeira...")

query_memories
vec: [0.0023313637357205153, 0.02363940328359604, -0.05854569375514984, 0.021730877459049225, 0.003310306230559945, 0.027877649292349815, -0.004932553973048925, 0.027403807267546654, -0.011885513551533222, -0.07460363954305649, -0.059703972190618515, -0.02708791382610798, -0.028904303908348083, 0.0009863462764769793, 0.005511692725121975, -0.016005298122763634, -0.042935263365507126, 0.0003393392835278064, 0.021757202222943306, 0.02417905628681183, 0.036038246005773544, 0.010852276347577572, -0.010069122537970543, -0.006334333680570126, 0.01955910585820675, 0.02132284827530384, 0.04219817742705345, 0.06038840860128403, 0.030009932816028595, 0.04988493397831917, 0.02363940328359604, -0.016623923555016518, -0.030720695853233337, -0.04309321194887161, -0.01811125874519348, -0.026916803792119026, 0.004902938846498728, -0.010115190409123898, -0.0009049047948792577, -0.022270528599619865, 0.01700562983751297, 0.018453476950526237, 0.0415663905441761, 0.019624916836619377, 0.02

In [46]:
_ = await chat(
    "Oh that's a cool idea. One time i took classes from this studio nearby. Wonder if they have any recs. "
)

query_memories
vec: [-0.0051300907507538795, 0.011447479948401451, -0.05803350731730461, 0.023498903959989548, -0.002355723874643445, 0.018186943605542183, -0.013032833114266396, 0.033820852637290955, -0.011859260499477386, -0.07335171848535538, -0.04727232828736305, -0.029016753658652306, -0.031048201024532318, 0.0022407688666135073, -0.002846428193151951, 0.0008801794610917568, -0.04513107240200043, -0.0025375934783369303, 0.02931872569024563, 0.035413067787885666, 0.04150741174817085, 0.010534701868891716, -0.00997193530201912, -0.004876160062849522, 0.02182433195412159, 0.01736338436603546, 0.04422515630722046, 0.06281015276908875, 0.021289018914103508, 0.05649619549512863, 0.023622438311576843, -0.017706533893942833, -0.03854259476065636, -0.03706018626689911, -0.024940133094787598, -0.02461070939898491, 0.0022270428016781807, -0.008489527739584446, 0.005476672202348709, -0.031240366399288177, 0.017528096213936806, 0.030059929937124252, 0.04381337761878967, 0.016608454287052155, 0

In [47]:
_ = await chat("Idk. Anyways - how are you doing?")

query_memories
vec: [0.0027088571805506945, -0.0008445552666671574, -0.055320847779512405, 0.02747531607747078, 0.013764102011919022, 0.025611013174057007, -0.009665281511843204, 0.02328394167125225, -0.006677109748125076, -0.06341271102428436, -0.05344332382082939, -0.024844137951731682, -0.033451661467552185, 0.012792284600436687, 2.1369540263549425e-05, -0.009301677346229553, -0.05188312754034996, -0.0016915895976126194, 0.008805852383375168, 0.035831619054079056, 0.04109397530555725, 0.005407798103988171, -0.009552895091474056, -0.003854213049635291, 0.023746712133288383, 0.01877523958683014, 0.04543079063296318, 0.06066253408789635, 0.0046838936395943165, 0.04952961206436157, 0.023350052535533905, -0.013208777643740177, -0.04363260045647621, -0.02718443237245083, -0.02747531607747078, -0.02198818512260914, -0.012455124408006668, -0.012666676193475723, 0.007470429874956608, -0.03675715997815132, 0.005642488598823547, 0.03786780685186386, 0.04061798378825188, 0.018471134826540947, 0

In [48]:
_ = await chat("My name is Ken btw")

query_memories
vec: [-0.004831930622458458, -0.001042598974891007, -0.05273180827498436, 0.047221291810274124, 0.01095965038985014, 0.012650996446609497, 0.004831930622458458, 0.033172208815813065, -0.01481291837990284, -0.05093134194612503, -0.05453227460384369, -0.028780164197087288, -0.01925952173769474, 0.02362428605556488, -0.004883080255240202, -0.005084268283098936, -0.06334364414215088, -0.006001550704240799, -0.0033161977771669626, 0.05734208971261978, 0.07763824611902237, -0.02492007613182068, -0.003542961087077856, 0.03991031274199486, 0.013585329055786133, -0.017008941620588303, 0.03603658452630043, 0.030171433463692665, -0.026897860690951347, 0.05322284623980522, 0.017336297780275345, -0.03611842542886734, -0.04863984137773514, 0.0021840871777385473, -0.011648464947938919, -0.015713151544332504, -0.03644578158855438, 0.0017314133001491427, 0.01291697472333908, -0.044984351843595505, -0.008081633597612381, 0.019136764109134674, 0.04135614261031151, 0.0005852364702150226, 0.

## Convo 2

Our memory is configured only to consider a thread "ready to process" if has been inactive for a minute.
We'll wait for things to populate

In [49]:
import asyncio

await asyncio.sleep(60)

CancelledError: 

In [50]:
thread_id_2 = uuid.uuid4()

In [51]:
chat2 = Chat(user_id, thread_id_2)

In [52]:
_ = await chat2("Remember me?")

query_memories
vec: [0.006665219087153673, -0.01612507924437523, -0.05111398547887802, 0.01189818512648344, -0.004237374756485224, 0.025319449603557587, -0.01880793459713459, 0.04024283215403557, -0.025319449603557587, 0.03135587275028229, -0.04613952711224556, -0.020275121554732323, 0.031160248443484306, 0.013071933761239052, 0.019450701773166656, 0.014867491088807583, -0.05097425356507301, 0.015929454937577248, -0.036833371967077255, -0.002008648356422782, -0.0012060622684657574, 0.039292655885219574, 0.024774493649601936, 0.009858096949756145, -0.0026199761778116226, -0.03297676518559456, -0.001933542313054204, -0.01914329268038273, 0.03851015493273735, -0.0404384583234787, -0.0024907239712774754, -0.03864988684654236, -0.0019545021932572126, -0.027443375438451767, 0.024704627692699432, -0.007929794490337372, 0.009243275970220566, 0.00025282768183387816, 0.0008484355639666319, 0.03001444600522518, 0.008495708927512169, -0.013085907325148582, 0.04166809841990471, 0.001231388770975172

In [44]:
_ = await chat2("wdy remember??")

I remember because I have a special memory book where I keep track of all the fun conversations and events we've shared together! It's like a digital scrapbook, and it helps me remember important details about our chats.

In [45]:
_ = await chat2("Oh planning is going alright!")

That's great to hear! I'm glad to know that the planning is going smoothly. Are there any new developments or updates that you'd like to share about the party? Maybe I can even offer some suggestions or ideas to make it an even more special celebration for Steve!